# Phase 3A — Figure 3: Participant PRE–POST Relative-Power Change

**Change score:** Δ = POST − PRE. Positive values indicate an increase in relative band power; negative values indicate a decrease. This figure is descriptive and does not perform new inferential tests.

For each task-condition, the figure displays participant change points, group mean change, and 95% t-based confidence intervals across five bands × five ROIs. A horizontal zero line marks no change.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import t

# Robust input discovery for local Mac / hosted environments
candidates = [Path.cwd()/'eeg_roi_power_long.csv', Path.home()/'Term Project'/'analysis'/'eeg_roi_power_long.csv', Path.home()/'eeg_roi_power_long.csv', Path('/mnt/data/eeg_roi_power_long.csv'), Path('/mnt/data/step2_v3_audit_test_outputs/eeg_roi_power_long.csv')]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find eeg_roi_power_long.csv. Place this notebook beside the CSV or in Term Project/analysis.')
print('Using:', csv_path)
df = pd.read_csv(csv_path)
required = {'subject_id','session','condition','band','roi','relative_power_roi','task','group'}
missing = required - set(df.columns)
if missing: raise ValueError(f'Missing required columns: {missing}')
print('Rows:', len(df), '| Subjects:', df.subject_id.nunique())

Using: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_power_long.csv
Rows: 8000 | Subjects: 63


In [4]:
# Pair PRE and POST within participant/task/condition/band/ROI
keys = ['subject_id','task','condition','band','roi','group']
sub = df[df.session.isin(['PRE','POST'])].copy()
if sub.duplicated(keys+['session']).any():
    raise ValueError('Duplicate participant/session rows exist for a figure key.')
pivot = sub.pivot(index=keys, columns='session', values='relative_power_roi').reset_index()
if not {'PRE','POST'}.issubset(pivot.columns):
    raise ValueError('PRE and POST columns not both present after pairing.')
pivot = pivot.dropna(subset=['PRE','POST'])
pivot['delta'] = pivot['POST'] - pivot['PRE']
print('Paired change records:', len(pivot), '| Missing pairs excluded from change plotting:', len(sub)-2*len(pivot))
print('Tasks/conditions:', sorted(pivot[['task','condition']].drop_duplicates().itertuples(index=False, name=None)))
# Validation: one change value per participant/key; no missing change
assert not pivot.duplicated(keys).any()
assert pivot.delta.notna().all()

Paired change records: 4000 | Missing pairs excluded from change plotting: 0
Tasks/conditions: [('Math', 'Activity1'), ('Math', 'Activity2'), ('Math', 'Baseline'), ('Reading', 'Activity'), ('Reading', 'Baseline')]


In [5]:
# Order and group summaries
band_order = ['delta','theta','alpha','beta','gamma']
roi_order = ['Frontal','Central','Temporal','Parietal','Occipital']
group_order = ['Control','Experiment']
summary = (pivot.groupby(['task','condition','band','roi','group'], observed=True)['delta']
           .agg(mean='mean', sd='std', n='count').reset_index())
summary['ci95'] = summary.apply(lambda r: t.ppf(0.975, r.n-1)*r.sd/np.sqrt(r.n) if r.n>1 and pd.notna(r.sd) else np.nan, axis=1)
summary.to_csv('figure3_prepost_change_summary.csv', index=False)
print('Summary rows:', len(summary))

Summary rows: 250


In [6]:
# Plot participant change, group mean, and 95% CI in 5×5 panels
out_dir = csv_path.parent/'phase3_figures'/'figure3_prepost_change'
out_dir.mkdir(parents=True, exist_ok=True)
condition_sets = [('Math','Baseline'),('Math','Activity1'),('Math','Activity2'),('Reading','Baseline'),('Reading','Activity')]
figures_created = []
for task_name, condition_name in condition_sets:
    plot_data = pivot[(pivot.task==task_name)&(pivot.condition==condition_name)]
    if plot_data.empty:
        print(f'SKIP (no paired data): {task_name} / {condition_name}')
        continue
    fig, axes = plt.subplots(5,5, figsize=(18,15), sharex=False, sharey=False)
    fig.suptitle(f'{task_name} — {condition_name}: Relative Power PRE–POST Change (Δ)', fontsize=16, y=0.995)
    for i, band_name in enumerate(band_order):
        for j, roi_name in enumerate(roi_order):
            ax=axes[i,j]
            cell=plot_data[(plot_data.band==band_name)&(plot_data.roi==roi_name)]
            for gi, group_name in enumerate(group_order):
                vals=cell.loc[cell.group==group_name,'delta'].dropna()
                if len(vals):
                    jitter=np.linspace(-0.08,0.08,len(vals)) if len(vals)>1 else np.array([0.])
                    ax.scatter(np.full(len(vals),gi)+jitter, vals, alpha=0.45, s=13, label=group_name if (i==0 and j==0) else None)
                    mean=vals.mean(); ci=t.ppf(0.975,len(vals)-1)*vals.std(ddof=1)/np.sqrt(len(vals)) if len(vals)>1 else np.nan
                    ax.errorbar(gi,mean,yerr=ci if np.isfinite(ci) else None, marker='D', markersize=4, capsize=3, linewidth=1.2)
            ax.axhline(0, linestyle='--', linewidth=0.7, alpha=0.7)
            ax.set_xticks([0,1],['Control','Experiment'])
            if i==0: ax.set_title(roi_name)
            if j==0: ax.set_ylabel(f'{band_name.capitalize()}\nΔ Relative power (%)')
            ax.grid(axis='y', alpha=0.2)
    handles, labels = axes[0,0].get_legend_handles_labels()
    if handles: fig.legend(handles,labels,loc='upper right')
    fig.tight_layout(rect=[0,0,1,0.97])
    safe_task=task_name.lower(); safe_cond=condition_name.lower().replace(' ','_')
    out=out_dir/f'figure3_{safe_task}_{safe_cond}_relative_power_change.png'
    fig.savefig(out,dpi=300,bbox_inches='tight')
    plt.close(fig)
    figures_created.append(str(out))
print('Figures created:', len(figures_created))
for f in figures_created: print(f)

Figures created: 5
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure3_prepost_change/figure3_math_baseline_relative_power_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure3_prepost_change/figure3_math_activity1_relative_power_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure3_prepost_change/figure3_math_activity2_relative_power_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure3_prepost_change/figure3_reading_baseline_relative_power_change.png
/Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure3_prepost_change/figure3_reading_activity_relative_power_change.png


In [7]:
# Final QC manifest
manifest = pd.DataFrame({'figure_file': figures_created})
manifest.to_csv(out_dir/'figure3_output_manifest.csv', index=False)
print('Conditions represented:', len(figures_created), '/', len(condition_sets))
print('Output folder:', out_dir)
print('Change score range:', float(pivot.delta.min()), 'to', float(pivot.delta.max()))

Conditions represented: 5 / 5
Output folder: /Users/ananyosen/Documents/Term Project/analysis/phase3_figures/figure3_prepost_change
Change score range: -65.80333909716893 to 90.47067129338078
